In [ ]:
import os
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    f1_score, precision_score, recall_score,
    roc_auc_score
)
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

# === CONFIG ===
DATA_DIR   = '/home/laura/Scriptie/bewerkte_data/THRawS'
THRESHOLDS = np.linspace(0.05, 0.95, 19)

In [ ]:
X_train = np.load(os.path.join(DATA_DIR, 'X_train_THRawS.npy'))
y_train = np.load(os.path.join(DATA_DIR, 'y_train_THRawS.npy'))
X_test  = np.load(os.path.join(DATA_DIR, 'X_test_THRawS.npy'))
y_test  = np.load(os.path.join(DATA_DIR, 'y_test_THRawS.npy'))

In [ ]:
# ==================================================
# === Majorty class classifier for not-resamples ===
# ==================================================

mc_model = DummyClassifier(strategy='prior', random_state=42)
mc_model.fit(X_train, y_train)

mc_pred = mc_model.predict(X_test)
mc_fire_idx = list(mc_model.classes_).index(1)
mc_probs = mc_model.predict_proba(X_test)[:, mc_fire_idx]

print("=== MAJORITY CLASS BASELINE ===")
print(f"Prior probability of fire (class 1): {mc_probs[0]:.4f}")
print(f"Accuracy: {accuracy_score(y_test, mc_pred):.3f}")
print(classification_report(y_test, mc_pred,
      target_names=['No event (0)', 'Event (1)'], zero_division=0))

## Resample variant, where we simulate thresholds by changing the amount of samples per class according to threshold

## Save resampling results

## Baselines for threshold sweep variant

In [ ]:
# Random Forest
rf = RandomForestClassifier(
    n_estimators=100, random_state=42,
    n_jobs=-1, class_weight='balanced'
)
rf.fit(X_train, y_train)
fire_idx_rf = list(rf.classes_).index(1)
rf_probs = rf.predict_proba(X_test)[:, fire_idx_rf]

print("=== RANDOM FOREST (threshold 0.5) ===")
rf_pred_05 = (rf_probs >= 0.5).astype(int)
print(classification_report(y_test, rf_pred_05,
      target_names=['No event (0)', 'Event (1)'], zero_division=0))

# XGBoost
xgb_ratio = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    random_state=42, n_jobs=-1,
    scale_pos_weight=xgb_ratio, eval_metric='logloss'
)
xgb.fit(X_train, y_train)
fire_idx_xgb = list(xgb.classes_).index(1)
xgb_probs = xgb.predict_proba(X_test)[:, fire_idx_xgb]

print("=== XGBOOST (threshold 0.5) ===")
xgb_pred_05 = (xgb_probs >= 0.5).astype(int)
print(classification_report(y_test, xgb_pred_05,
      target_names=['No event (0)', 'Event (1)'], zero_division=0))

In [ ]:
rf_results, xgb_results = [], []

for t in THRESHOLDS:
    rf_pred  = (rf_probs  >= t).astype(int)
    xgb_pred = (xgb_probs >= t).astype(int)

    rf_results.append({
        'threshold': t,
        'probs':     rf_probs,
        'preds':     rf_pred,
        'f1':        f1_score(y_test, rf_pred, zero_division=0),
        'f1_wtd':    f1_score(y_test, rf_pred, average='weighted', zero_division=0),
        'precision': precision_score(y_test, rf_pred, zero_division=0),
        'recall':    recall_score(y_test, rf_pred, zero_division=0),
        'accuracy':  accuracy_score(y_test, rf_pred),
        'auroc':     roc_auc_score(y_test, rf_probs),
    })
    xgb_results.append({
        'threshold': t,
        'probs':     xgb_probs,
        'preds':     xgb_pred,
        'f1':        f1_score(y_test, xgb_pred, zero_division=0),
        'f1_wtd':    f1_score(y_test, xgb_pred, average='weighted', zero_division=0),
        'precision': precision_score(y_test, xgb_pred, zero_division=0),
        'recall':    recall_score(y_test, xgb_pred, zero_division=0),
        'accuracy':  accuracy_score(y_test, xgb_pred),
        'auroc':     roc_auc_score(y_test, xgb_probs),
    })

print(f"{len(THRESHOLDS)} thresholds evaluated")

In [ ]:
for name, results in [('Random Forest', rf_results), ('XGBoost', xgb_results)]:
    best = max(results, key=lambda x: x['f1'])
    print(f"=== {name} ===")
    print(f"  Best threshold: {best['threshold']:.2f}")
    print(f"  F1:        {best['f1']:.3f}")
    print(f"  F1 wtd:    {best['f1_wtd']:.3f}")
    print(f"  Precision: {best['precision']:.3f}")
    print(f"  Recall:    {best['recall']:.3f}")
    print(f"  Accuracy:  {best['accuracy']:.3f}")
    print(f"  AUROC:     {best['auroc']:.3f}")
    print()

# Full table for Random Forest
print(f"{'Threshold':>10} {'F1':>8} {'Precision':>10} {'Recall':>8} {'Accuracy':>10}")
print("-" * 50)
for r in rf_results:
    print(f"  {r['threshold']:>8.2f} {r['f1']:>8.3f} {r['precision']:>10.3f} "
          f"{r['recall']:>8.3f} {r['accuracy']:>10.3f}")

In [ ]:
print(f"mc_model.classes_: {mc_model.classes_}")
print(f"Prior per klasse: {mc_model.predict_proba(X_test[:1])}")

## Save sweep results

## Cost-sensitive variant, where we change the weight of the class to "simulate" thresholds

In [ ]:
rf_results_cs, xgb_results_cs = [], []

for t in THRESHOLDS:
    #pos_weight = np.clip((1 - t) / t, 0.1, 10)
    pos_weight = np.clip((1 - t) / t, 0.3, 3)

    # Random Forest
    rf_cs = RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1,
        class_weight={0: 1.0, 1: pos_weight}
    )
    rf_cs.fit(X_train, y_train)
    fire_idx = list(rf_cs.classes_).index(1)
    rf_prob_cs = rf_cs.predict_proba(X_test)[:, fire_idx]
    rf_pred_cs = (rf_prob_cs >= 0.5).astype(int)

    rf_results_cs.append({
        'threshold': t,
        'pos_weight': pos_weight,
        'f1':        f1_score(y_test, rf_pred_cs, zero_division=0),
        'precision': precision_score(y_test, rf_pred_cs, zero_division=0),
        'recall':    recall_score(y_test, rf_pred_cs, zero_division=0),
        'accuracy':  accuracy_score(y_test, rf_pred_cs),
        'auroc':     roc_auc_score(y_test, rf_prob_cs),
    })

    # XGBoost
    xgb_cs = XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        random_state=42, n_jobs=-1,
        scale_pos_weight=pos_weight, eval_metric='logloss'
    )
    xgb_cs.fit(X_train, y_train)
    fire_idx_xgb = list(xgb_cs.classes_).index(1)
    xgb_prob_cs = xgb_cs.predict_proba(X_test)[:, fire_idx_xgb]
    xgb_pred_cs = (xgb_prob_cs >= 0.5).astype(int)

    xgb_results_cs.append({
        'threshold': t,
        'pos_weight': pos_weight,
        'f1':        f1_score(y_test, xgb_pred_cs, zero_division=0),
        'precision': precision_score(y_test, xgb_pred_cs, zero_division=0),
        'recall':    recall_score(y_test, xgb_pred_cs, zero_division=0),
        'accuracy':  accuracy_score(y_test, xgb_pred_cs),
        'auroc':     roc_auc_score(y_test, xgb_prob_cs),
    })

print(f"{'Threshold':>10} {'PosWeight':>10} {'F1':>8} {'Precision':>10} {'Recall':>8} {'Accuracy':>10}")
print("-" * 65)
for r in rf_results_cs:
    print(f"  {r['threshold']:>8.2f} {r['pos_weight']:>10.3f} {r['f1']:>8.3f} "
          f"{r['precision']:>10.3f} {r['recall']:>8.3f} {r['accuracy']:>10.3f}")

In [ ]:
base_idx_0 = np.where(y_train == 0)[0]
base_idx_1 = np.where(y_train == 1)[0]
n0, n1 = len(base_idx_0), len(base_idx_1)

mc_results_resample = []

for t in THRESHOLDS:
    pos_weight = (1 - t) / t
    target_n1 = max(int(round(n1 * pos_weight)), 5)
    
    # Calculate the new prior for class 1 (fire) after resampling
    total_samples = n0 + target_n1
    prior_fire = target_n1 / total_samples
    
    if prior_fire > 0.5:
        mc_pred = np.ones(len(y_test), dtype=int)
    else:
        mc_pred = np.zeros(len(y_test), dtype=int)

    mc_prob = np.full(len(y_test), prior_fire)
    
    mc_results_resample.append({
        'threshold':  t,
        'prior_fire': prior_fire,
        'majority':   'fire' if prior_fire > 0.5 else 'no-fire',
        'f1':        f1_score(y_test, mc_pred, zero_division=0),
        'auroc':     roc_auc_score(y_test, mc_prob), # Always the same as prior_fire, since it's a constant probability for all samples (0.5)
    })

print(f"{'Threshold':>10} {'Prior fire':>12} {'Majority':>10} {'F1':>8} {'AUROC':>8}")
print("-" * 55)
for r in mc_results_resample:
    print(f"  {r['threshold']:>8.2f} {r['prior_fire']:>12.3f} {r['majority']:>10} "
          f"{r['f1']:>8.3f} {r['auroc']:>8.3f}")

In [ ]:
# ========================================
# RESAMPLING PER THRESHOLD — Random Forest
# ========================================
#
# Goal is to simulate 19 "different models" by creating a different trainingset
# for each threshold, instead of training one model and shifting the cutoff afterwards
#
# How to do this is by changing the ratio of event/no-event in the training data.
# A model that is trained on many event samples learnc to call "event" faster.
# This simulates a lower threshold. A model with fewer event samples will be more
# cautious — this simulates a higher threshold.
#

base_idx_0 = np.where(y_train == 0)[0] # No-event samples - constant over thresholds
base_idx_1 = np.where(y_train == 1)[0] # Event samples - changes over thresholds
n0, n1 = len(base_idx_0), len(base_idx_1)

rng_rf = np.random.default_rng(seed=42) 

rf_results_resample = []

for t in THRESHOLDS:
    # Step 1: calculate how many event samples we want, based on the threshold.
    # Use the resample factor (pos_weight) to determine the target number of event samples.
    #   - low t → high pos_weight →  more event samples (oversampling)
    #   - high t → low pos_weight → fewer event samples (undersampling)

    pos_weight = (1 - t) / t
    target_n1 = max(int(round(n1 * pos_weight)), 5)  # At least 5 samples, otherwise unstable

    # Step 2: draw a new selection of event samples from the ORIGINAL event samples.
    # With replacement (replace=True) if we need more than there are
    # (duplicate), without replacement if we need fewer (omit).
    sampled_idx_1 = rng_rf.choice(base_idx_1, size=target_n1, replace=(target_n1 >= n1))

    # Step 3: assemble the new training set — all no-event samples
    # plus the (adjusted amount of) event samples, shuffled so that the
    # order does not reveal the class.
    resampled_idx = np.concatenate([base_idx_0, sampled_idx_1])
    rng_rf.shuffle(resampled_idx)
    X_train_t = X_train[resampled_idx]
    y_train_t = y_train[resampled_idx]

    # Step 4: train a new model on this resampled data.
    rf_t = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_t.fit(X_train_t, y_train_t)

    fire_idx = list(rf_t.classes_).index(1)
    probs_t = rf_t.predict_proba(X_test)[:, fire_idx]

    # Step 5: always evaluate on the neutral threshold of 0.5.
    # The threshold-simulation is already implemented in the training data,
    # so the cutoff value when making predictions must remain neutral, it otherwise
    # amplifies the effect unnecessarily.
    pred_t = (probs_t >= 0.5).astype(int)

    rf_results_resample.append({
        'threshold':    t,             # simulated threshold
        'n1_resampled': target_n1,     # how many fire samples used
        'f1':        f1_score(y_test, pred_t, zero_division=0),
        'precision': precision_score(y_test, pred_t, zero_division=0),
        'recall':    recall_score(y_test, pred_t, zero_division=0),
        'accuracy':  accuracy_score(y_test, pred_t),
        'auroc':     roc_auc_score(y_test, probs_t),
    })

print(f"{'Threshold':>10} {'N(brand)':>10} {'F1':>8} {'Precision':>10} {'Recall':>8} {'Accuracy':>10}")
print("-" * 60)
for r in rf_results_resample:
    print(f"  {r['threshold']:>8.2f} {r['n1_resampled']:>10} {r['f1']:>8.3f} "
          f"{r['precision']:>10.3f} {r['recall']:>8.3f} {r['accuracy']:>10.3f}")

In [ ]:
# ==================================
# RESAMPLING PER THRESHOLD — XGBoost
# ==================================

xgb_results_resample = []

rng_xgb = np.random.default_rng(seed=42)

for t in THRESHOLDS:
    pos_weight = (1 - t) / t
    target_n1 = max(int(round(n1 * pos_weight)), 5)

    sampled_idx_1 = rng_xgb.choice(base_idx_1, size=target_n1, replace=(target_n1 >= n1))

    resampled_idx = np.concatenate([base_idx_0, sampled_idx_1])
    rng_xgb.shuffle(resampled_idx)

    X_train_t = X_train[resampled_idx]
    y_train_t = y_train[resampled_idx]

    xgb_t = XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        random_state=42, n_jobs=-1, eval_metric='logloss'
    )
    xgb_t.fit(X_train_t, y_train_t)

    fire_idx = list(xgb_t.classes_).index(1)
    probs_t = xgb_t.predict_proba(X_test)[:, fire_idx]
    pred_t  = (probs_t >= 0.5).astype(int)

    xgb_results_resample.append({
        'threshold':    t,
        'n1_resampled': target_n1,
        'f1':        f1_score(y_test, pred_t, zero_division=0),
        'precision': precision_score(y_test, pred_t, zero_division=0),
        'recall':    recall_score(y_test, pred_t, zero_division=0),
        'accuracy':  accuracy_score(y_test, pred_t),
        'auroc':     roc_auc_score(y_test, probs_t),
    })

print(f"{'Threshold':>10} {'N(brand)':>10} {'F1':>8} {'Precision':>10} {'Recall':>8} {'Accuracy':>10}")
print("-" * 60)
for r in xgb_results_resample:
    print(f"  {r['threshold']:>8.2f} {r['n1_resampled']:>10} {r['f1']:>8.3f} "
          f"{r['precision']:>10.3f} {r['recall']:>8.3f} {r['accuracy']:>10.3f}")

In [ ]:
best_rf  = max(rf_results_resample,  key=lambda x: x['f1'])
best_xgb = max(xgb_results_resample, key=lambda x: x['f1'])

print(f"RF  — best threshold: {best_rf['threshold']:.2f}, F1: {best_rf['f1']:.3f}, AUROC: {best_rf['auroc']:.3f}")
print(f"XGB — best threshold: {best_xgb['threshold']:.2f}, F1: {best_xgb['f1']:.3f}, AUROC: {best_xgb['auroc']:.3f}")

best_rf_auc  = max(rf_results_resample,  key=lambda x: x['auroc'])
best_xgb_auc = max(xgb_results_resample, key=lambda x: x['auroc'])

print(f"RF  — best threshold (AUROC): {best_rf_auc['threshold']:.2f}, AUROC: {best_rf_auc['auroc']:.3f}")
print(f"XGB — best threshold (AUROC): {best_xgb_auc['threshold']:.2f}, AUROC: {best_xgb_auc['auroc']:.3f}")

In [ ]:
# Plotting the results, both in F1 score and AUROC

thresholds_plot = [r['threshold'] for r in rf_results_resample]
rf_f1   = [r['f1']   for r in rf_results_resample]
xgb_f1  = [r['f1']   for r in xgb_results_resample]
rf_auc  = [r['auroc'] for r in rf_results_resample]
xgb_auc = [r['auroc'] for r in xgb_results_resample]
mc_f1_resample   = [r['f1']   for r in mc_results_resample]
mc_auroc_resample = [r['auroc'] for r in mc_results_resample]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plotting F1 score
axes[0].plot(thresholds_plot, rf_f1,  marker='o', label='Random Forest', linewidth=2, color='royalblue')
axes[0].plot(thresholds_plot, xgb_f1, marker='s', label='XGBoost',       linewidth=2, color='darkorange')
axes[0].plot(thresholds_plot, mc_f1_resample, label='Majority Class', linewidth=2, color='darkgreen')
axes[0].axvline(best_rf['threshold'],  color='royalblue',  linestyle='-.', alpha=0.5,
                label=f"RF best (t={best_rf['threshold']:.2f})")
axes[0].axvline(best_xgb['threshold'], color='darkorange', linestyle='-.', alpha=0.5,
                label=f"XGB best (t={best_xgb['threshold']:.2f})")
axes[0].axvline(0.5, color='gray', linestyle=':', label='Threshold 0.5')
axes[0].set_title('F1 Score', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Simulated Threshold (via Resampling)')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.05)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plotting AUROC
axes[1].plot(thresholds_plot, rf_auc,  marker='o', label='Random Forest', linewidth=2, color='royalblue')
axes[1].plot(thresholds_plot, xgb_auc, marker='s', label='XGBoost',       linewidth=2, color='darkorange')
axes[1].plot(thresholds_plot, mc_auroc_resample,
             color='darkgreen', linewidth=2, label='Majority Class')
axes[1].axvline(best_rf_auc['threshold'],  color='royalblue',  linestyle='-.', alpha=0.5,
                label=f"RF best (t={best_rf_auc['threshold']:.2f})")
axes[1].axvline(best_xgb_auc['threshold'], color='darkorange', linestyle='-.', alpha=0.5,
                label=f"XGB best (t={best_xgb_auc['threshold']:.2f})")
axes[1].axvline(0.5, color='gray', linestyle=':', label='Threshold 0.5')
axes[1].set_title('AUROC', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Simulated Threshold (via Resampling)')
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle('THRawS: Resampling per Threshold', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
np.save(os.path.join(DATA_DIR, 'y_test_THRawS.npy'), y_test)

np.save(os.path.join(DATA_DIR, 'y_prob_majority_THRawS.npy'), mc_probs)

np.save(os.path.join(DATA_DIR, 'results_rf_THRawS.npy'),  rf_results,  allow_pickle=True)
np.save(os.path.join(DATA_DIR, 'results_xgb_THRawS.npy'), xgb_results, allow_pickle=True)

np.save(os.path.join(DATA_DIR, 'y_prob_rf_THRawS.npy'),  rf_probs)
np.save(os.path.join(DATA_DIR, 'y_prob_xgb_THRawS.npy'), xgb_probs)

In [ ]:
np.save(os.path.join(DATA_DIR, 'results_rf_resample_THRawS.npy'),  rf_results_resample,  allow_pickle=True)
np.save(os.path.join(DATA_DIR, 'results_xgb_resample_THRawS.npy'), xgb_results_resample, allow_pickle=True)